In [33]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


In [34]:

DATASET_FILE = "eye_dataset_v2.csv"

CLEAN_DATASET_FILE = "eye_dataset_clean.csv"

FEATURES = [
    "left_ear",
    "right_ear",
    "ear",
    "left_eye_width",
    "left_eye_height_1",
    "left_eye_height_2",
    "right_eye_width",
    "right_eye_height_1",
    "right_eye_height_2"
]
TARGET = "label"

TEST_SIZE = 0.20
RANDOM_STATE = 42

CONTAMINATION = 0.05


print()
print("=" * 70)
print("ISOLATION FOREST + RANDOM FOREST")
print("=" * 70)

print()
print("Loading dataset...")




ISOLATION FOREST + RANDOM FOREST

Loading dataset...


In [35]:

df = pd.read_csv(DATASET_FILE)


print()
print("ORIGINAL DATASET")
print("-" * 70)

print(f"Dataset shape : {df.shape}")

print()
print("Class distribution:")
print(df[TARGET].value_counts())
df


ORIGINAL DATASET
----------------------------------------------------------------------
Dataset shape : (1000, 10)

Class distribution:
label
OPEN      500
CLOSED    500
Name: count, dtype: int64


,left_ear,right_ear,ear,left_eye_width,left_eye_height_1,left_eye_height_2,right_eye_width,right_eye_height_1,right_eye_height_2,label
0,0.427398,0.432328,0.429863,0.039186,0.017069,0.016427,0.035552,0.014775,0.015965,OPEN
1,0.427019,0.435539,0.431279,0.039078,0.016954,0.016420,0.035660,0.014906,0.016156,OPEN
2,0.427494,0.447180,0.437337,0.039127,0.017017,0.016436,0.035825,0.015400,0.016641,OPEN
3,0.433946,0.444521,0.439234,0.039218,0.017375,0.016662,0.035812,0.015204,0.016635,OPEN
4,0.423431,0.440666,0.432049,0.039083,0.016941,0.016157,0.036280,0.015251,0.016724,OPEN
...,...,...,...,...,...,...,...,...,...,...
995,0.156201,0.219984,0.188093,0.034444,0.005484,0.005276,0.030398,0.006173,0.007201,CLOSED
996,0.133958,0.220235,0.177097,0.034373,0.004669,0.004540,0.031537,0.006374,0.007517,CLOSED
997,0.157326,0.234718,0.196022,0.033819,0.005462,0.005179,0.030698,0.006550,0.007861,CLOSED
998,0.165653,0.212419,0.189036,0.033581,0.005775,0.005351,0.030503,0.005807,0.007151,CLOSED


### Missing Feature

In [36]:

missing_features = [
    feature
    for feature in FEATURES
    if feature not in df.columns
]

if missing_features:

    raise ValueError(
        f"Missing features: {missing_features}"
    )


if df[FEATURES].isnull().sum().sum() > 0:

    raise ValueError(
        "Dataset contains missing values."
    )

missing_features

[]

####    ISOLATION FOREST


In [37]:
print()
print("=" * 70)
print("ISOLATION FOREST")
print("=" * 70)

X_all = df[FEATURES]


isolation_forest = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


isolation_prediction = isolation_forest.fit_predict(
    X_all
)


# Isolation Forest:
#  1  = normal
# -1  = anomaly

df["is_outlier"] = (
    isolation_prediction == -1
)
isolation_forest


ISOLATION FOREST


,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


### OUTLIER SUMMARY


In [38]:
outlier_count = df["is_outlier"].sum()

normal_count = (
    (~df["is_outlier"]).sum()
)


print()
print("OUTLIER RESULTS")
print("-" * 70)

print(
    f"Original samples : {len(df)}"
)

print(
    f"Normal samples   : {normal_count}"
)

print(
    f"Outliers         : {outlier_count}"
)

print(
    f"Outlier %        : {outlier_count / len(df) * 100:.2f}%"
)


print()
print("OUTLIERS BY CLASS")
print("-" * 70)

print(
    pd.crosstab(
        df[TARGET],
        df["is_outlier"]
    )
)



OUTLIER RESULTS
----------------------------------------------------------------------
Original samples : 1000
Normal samples   : 950
Outliers         : 50
Outlier %        : 5.00%

OUTLIERS BY CLASS
----------------------------------------------------------------------
is_outlier  False  True 
label                   
CLOSED        474     26
OPEN          476     24


#### Cleaning


In [39]:

# ============================================================
# CREATE CLEAN DATASET
# ============================================================

df_clean = df[
    df["is_outlier"] == False
].copy()


# Remove helper column

df_clean.drop(
    columns=["is_outlier"],
    inplace=True
)


# ============================================================
# SAVE CLEAN DATASET
# ============================================================

df_clean.to_csv(
    CLEAN_DATASET_FILE,
    index=False
)


print()
print("CLEAN DATASET")
print("-" * 70)

print(
    f"Clean dataset shape : {df_clean.shape}"
)

print()
print("Clean class distribution:")

print(
    df_clean[TARGET].value_counts()
)

print()
print(
    f"Saved to: {CLEAN_DATASET_FILE}"
)



CLEAN DATASET
----------------------------------------------------------------------
Clean dataset shape : (950, 10)

Clean class distribution:
label
OPEN      476
CLOSED    474
Name: count, dtype: int64

Saved to: eye_dataset_clean.csv


In [40]:
X = df_clean[FEATURES]

y = df_clean[TARGET]


## TRAIN / TEST SPLIT


In [41]:

print()
print("TRAIN / TEST SPLIT")
print("-" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


print(
    f"Training samples : {len(X_train)}"
)

print(
    f"Testing samples  : {len(X_test)}"
)




TRAIN / TEST SPLIT
----------------------------------------------------------------------
Training samples : 760
Testing samples  : 190


# RANDOM FOREST

In [43]:

print()
print("TRAINING RANDOM FOREST")
print("-" * 70)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


model.fit(
    X_train,
    y_train
)


# PREDICTION


y_pred = model.predict(
    X_test
)



# METRICS


accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label="CLOSED"
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label="CLOSED"
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="CLOSED"
)


# RESULTS


print()
print("=" * 70)
print("RANDOM FOREST RESULTS AFTER OUTLIER REMOVAL")
print("=" * 70)

print()
print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Precision CLOSED  : {precision:.4f}"
)

print(
    f"Recall CLOSED     : {recall:.4f}"
)

print(
    f"F1 CLOSED         : {f1:.4f}"
)


# CONFUSION MATRIX


cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[
        "CLOSED",
        "OPEN"
    ]
)


print()
print("CONFUSION MATRIX")
print("-" * 70)

print(
    "                 Predicted"
)

print(
    "                 CLOSED    OPEN"
)

print(
    f"Actual CLOSED    {cm[0,0]:6d}   {cm[0,1]:6d}"
)

print(
    f"Actual OPEN      {cm[1,0]:6d}   {cm[1,1]:6d}"
)


# CLASSIFICATION REPORT


print()
print("CLASSIFICATION REPORT")
print("-" * 70)

print(
    classification_report(
        y_test,
        y_pred
    )
)


# FEATURE IMPORTANCE


importance = pd.DataFrame({

    "feature": FEATURES,

    "importance": model.feature_importances_

})


importance = importance.sort_values(
    by="importance",
    ascending=False
)


print()
print("FEATURE IMPORTANCE")
print("-" * 70)

print(
    importance.to_string(
        index=False
    )
)


importance.to_csv(
    "random_forest_clean_feature_importance.csv",
    index=False
)



print()
print("=" * 70)
print("CLEAN RANDOM FOREST ANALYSIS COMPLETE")
print("=" * 70)

print()
print(
    "Clean dataset:"
)

print(
    CLEAN_DATASET_FILE
)

print()
print(
    "Feature importance:"
)

print(
    "random_forest_clean_feature_importance.csv"
)

print()


TRAINING RANDOM FOREST
----------------------------------------------------------------------

RANDOM FOREST RESULTS AFTER OUTLIER REMOVAL

Accuracy          : 0.9842
Precision CLOSED  : 0.9792
Recall CLOSED     : 0.9895
F1 CLOSED         : 0.9843

CONFUSION MATRIX
----------------------------------------------------------------------
                 Predicted
                 CLOSED    OPEN
Actual CLOSED        94        1
Actual OPEN           2       93

CLASSIFICATION REPORT
----------------------------------------------------------------------
              precision    recall  f1-score   support

      CLOSED       0.98      0.99      0.98        95
        OPEN       0.99      0.98      0.98        95

    accuracy                           0.98       190
   macro avg       0.98      0.98      0.98       190
weighted avg       0.98      0.98      0.98       190


FEATURE IMPORTANCE
----------------------------------------------------------------------
           feature  impor